# IE492 — Phase 2.1: Beceri Bazlı Eşleştirme (Terk Yok / $\theta = 0$)

**Model Özeti:**
- Oyuncular Poisson($\lambda$) süreciyle gelir.
- Her oyuncu $p$ olasılıkla **İyi (Good)**, $1-p$ olasılıkla **Kötü (Bad)** beceri seviyesindedir.
- **Homojen Öncelik:** İyiler iyilerle ($G-G$), kötüler kötülerle ($B-B$) eşleşir.
- **Çapraz Eşleşme Eşiği ($\tau$):** Kendi seviyesinden oyuncu bulamayan bir oyuncu $\tau$ süre bekledikten sonra "esnek" hale gelir ve karşı seviyedeki oyuncuyla eşleşebilir ($G-B$).
- **Terk:** Bu aşamada terk etme yoktur ($\theta = 0$).

**Temel Hedef:** $\tau$ bekleme eşiğinin bekleme süresi, kuyruk uzunluğu ve maç kalitesi (çapraz maç oranı) üzerindeki etkisini incelemek.

In [ ]:
import heapq
from collections import deque
import numpy as np
import matplotlib.pyplot as plt

GOOD, BAD = 0, 1
TIER_NAME = {GOOD: "İyi (Good)", BAD: "Kötü (Bad)"}

WAITING = 0     # Kendi sınıfını bekliyor (< tau)
FLEXIBLE = 1    # tau süresini aştı, cross-match havuzunda
MATCHED = 2     # Eşleşti

EVT_ARRIVAL = 1
EVT_TIMEOUT = 2

MATCH_HOMOGENEOUS = 0  # G-G veya B-B
MATCH_CROSS = 1        # G-B

## Simülasyon Fonksiyonları

In [ ]:
def simulate_phase2_1(lam=1.0, p=0.6, tau=3.0, T=50_000, warmup=1_000, seed=42):
    rng = np.random.default_rng(seed)
    arrival, tier, end, status, partner_id, match_type = [], [], [], [], [], []
    queues = {GOOD: deque(), BAD: deque()}
    n_waiting = {GOOD: 0, BAD: 0}
    events = []
    area = {GOOD: 0.0, BAD: 0.0}
    last_t = 0.0
    trace_t, trace_q_good, trace_q_bad, trace_q_total = [0.0], [0], [0], [0]

    def advance(to):
        nonlocal last_t
        start = max(last_t, warmup)
        if to > start:
            for c in (GOOD, BAD):
                area[c] += n_waiting[c] * (to - start)
        last_t = to

    def record_trace(t):
        trace_t.append(t)
        trace_q_good.append(n_waiting[GOOD])
        trace_q_bad.append(n_waiting[BAD])
        trace_q_total.append(n_waiting[GOOD] + n_waiting[BAD])

    t_first = rng.exponential(1.0 / lam)
    heapq.heappush(events, (t_first, EVT_ARRIVAL, -1))

    while events:
        evt_t, evt_type, pid = heapq.heappop(events)
        if evt_t > T:
            advance(T)
            break
        advance(evt_t)

        if evt_type == EVT_ARRIVAL:
            t_next = evt_t + rng.exponential(1.0 / lam)
            if t_next <= T:
                heapq.heappush(events, (t_next, EVT_ARRIVAL, -1))

            c = GOOD if rng.random() < p else BAD
            other = 1 - c
            curr_id = len(arrival)
            arrival.append(evt_t); tier.append(c); end.append(np.nan)
            status.append(WAITING); partner_id.append(-1); match_type.append(-1)

            # 1. Kendi sınıfında bekleyen var mı?
            matched_partner = None
            while queues[c]:
                cand = queues[c].popleft()
                if status[cand] in (WAITING, FLEXIBLE):
                    matched_partner = cand
                    break

            if matched_partner is not None:
                status[matched_partner] = MATCHED; end[matched_partner] = evt_t
                partner_id[matched_partner] = curr_id; match_type[matched_partner] = MATCH_HOMOGENEOUS
                status[curr_id] = MATCHED; end[curr_id] = evt_t
                partner_id[curr_id] = matched_partner; match_type[curr_id] = MATCH_HOMOGENEOUS
                n_waiting[c] -= 1
                record_trace(evt_t)
                continue

            # 2. Karşı sınıfta FLEXIBLE (>= tau) oyuncu var mı?
            cross_partner = None
            for cand in queues[other]:
                if status[cand] == FLEXIBLE:
                    cross_partner = cand
                    break

            if cross_partner is not None:
                queues[other].remove(cross_partner)
                status[cross_partner] = MATCHED; end[cross_partner] = evt_t
                partner_id[cross_partner] = curr_id; match_type[cross_partner] = MATCH_CROSS
                status[curr_id] = MATCHED; end[curr_id] = evt_t
                partner_id[curr_id] = cross_partner; match_type[curr_id] = MATCH_CROSS
                n_waiting[other] -= 1
                record_trace(evt_t)
                continue

            # Eşleşme yok -> Kuyruğa gir
            queues[c].append(curr_id)
            n_waiting[c] += 1
            heapq.heappush(events, (evt_t + tau, EVT_TIMEOUT, curr_id))
            record_trace(evt_t)

        elif evt_type == EVT_TIMEOUT:
            if status[pid] == WAITING:
                status[pid] = FLEXIBLE
                c = tier[pid]; other = 1 - c
                cross_partner = None
                while queues[other]:
                    cand = queues[other].popleft()
                    if status[cand] in (WAITING, FLEXIBLE):
                        cross_partner = cand
                        break

                if cross_partner is not None:
                    queues[c].remove(pid)
                    status[pid] = MATCHED; end[pid] = evt_t
                    partner_id[pid] = cross_partner; match_type[pid] = MATCH_CROSS
                    status[cross_partner] = MATCHED; end[cross_partner] = evt_t
                    partner_id[cross_partner] = pid; match_type[cross_partner] = MATCH_CROSS
                    n_waiting[c] -= 1; n_waiting[other] -= 1
                    record_trace(evt_t)

    players = {
        "arrival": np.array(arrival), "tier": np.array(tier),
        "end": np.array(end), "status": np.array(status),
        "partner_id": np.array(partner_id), "match_type": np.array(match_type),
    }
    span = T - warmup
    avg_queue = {
        GOOD: area[GOOD] / span,
        BAD: area[BAD] / span,
        "total": (area[GOOD] + area[BAD]) / span
    }
    trace = {
        "t": np.array(trace_t),
        "good": np.array(trace_q_good),
        "bad": np.array(trace_q_bad),
        "total": np.array(trace_q_total),
    }
    return players, avg_queue, trace

def compute_kpis_phase2_1(players, avg_queue, warmup=1_000):
    kpis = {}
    for c in (GOOD, BAD):
        sel = (players["tier"] == c) & (players["arrival"] >= warmup) & (players["status"] == MATCHED)
        w = players["end"][sel] - players["arrival"][sel]
        mt = players["match_type"][sel]
        kpis[TIER_NAME[c]] = {
            "oyuncu sayisi": int(sel.sum()),
            "ort. bekleme": float(w.mean()) if len(w) > 0 else 0.0,
            "medyan bekleme": float(np.median(w)) if len(w) > 0 else 0.0,
            "p90 bekleme": float(np.percentile(w, 90)) if len(w) > 0 else 0.0,
            "beklemeden eslesme orani": float((w == 0).mean()) if len(w) > 0 else 0.0,
            "ort. kuyruk uzunlugu": avg_queue[c],
            "homojen mac sayisi": int((mt == MATCH_HOMOGENEOUS).sum()),
            "capraz mac sayisi": int((mt == MATCH_CROSS).sum()),
        }

    sel_all = (players["arrival"] >= warmup) & (players["status"] == MATCHED)
    w_all = players["end"][sel_all] - players["arrival"][sel_all]
    mt_all = players["match_type"][sel_all]
    total_matched = len(w_all)
    cross_matched = int((mt_all == MATCH_CROSS).sum())
    cross_ratio = cross_matched / total_matched if total_matched > 0 else 0.0

    kpis["genel"] = {
        "toplam eslesen oyuncu": total_matched,
        "ort. bekleme (genel)": float(w_all.mean()) if len(w_all) > 0 else 0.0,
        "ort. toplam kuyruk": avg_queue["total"],
        "capraz eslesme orani (%)": cross_ratio * 100.0,
        "homojen eslesme orani (%)": (1.0 - cross_ratio) * 100.0,
    }
    return kpis

def print_kpis_phase2_1(kpis):
    print("=" * 60)
    print(f"{'KPI':<30}{'İyi (Good)':>14}{'Kötü (Bad)':>14}")
    print("-" * 60)
    for k in ["oyuncu sayisi", "ort. bekleme", "medyan bekleme", "p90 bekleme",
              "beklemeden eslesme orani", "ort. kuyruk uzunlugu", "capraz mac sayisi"]:
        print(f"{k:<30}{kpis[TIER_NAME[GOOD]][k]:>14.3f}{kpis[TIER_NAME[BAD]][k]:>14.3f}")
    print("-" * 60)
    print(f"Genel Ortalama Bekleme Süresi : {kpis['genel']['ort. bekleme (genel)']:>10.3f}")
    print(f"Genel Ortalama Toplam Kuyruk  : {kpis['genel']['ort. toplam kuyruk']:>10.3f}")
    print(f"Çapraz Eşleşme Oranı (G-B)    : %{kpis['genel']['capraz eslesme orani (%)']:>9.2f}")
    print(f"Homojen Eşleşme Oranı (G-G,B-B): %{kpis['genel']['homojen eslesme orani (%)']:>8.2f}")
    print("=" * 60)

## Tek Senaryo Koşumu
Parametreler: $\lambda = 1.0, p = 0.6, \tau = 3.0$.

In [ ]:
LAM = 1.0
P = 0.6
TAU = 3.0
T = 50_000
WARMUP = 1_000
SEED = 42

players, avg_queue, trace = simulate_phase2_1(lam=LAM, p=P, tau=TAU, T=T, warmup=WARMUP, seed=SEED)
kpis = compute_kpis_phase2_1(players, avg_queue, warmup=WARMUP)
print_kpis_phase2_1(kpis)

## Kuyruk ve Bekleme Dağılımı Grafikleri

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4))
tt = trace["t"]
window = (tt >= WARMUP) & (tt <= WARMUP + 2000)

# 1. Sınıf bazında anlık kuyruklar
axs[0].step(tt[window], trace["good"][window], label="İyi Kuyruğu", alpha=0.8)
axs[0].step(tt[window], trace["bad"][window], label="Kötü Kuyruğu", alpha=0.8)
axs[0].set_xlabel("Zaman")
axs[0].set_ylabel("Bekleyen Oyuncu Sayısı")
axs[0].set_title("Sınıf Bazında Kuyruklar")
axs[0].legend()

# 2. Toplam anlık kuyruk ve ortalama
axs[1].step(tt[window], trace["total"][window], color="tab:purple", label="Anlık Toplam")
axs[1].axhline(kpis["genel"]["ort. toplam kuyruk"], color="red", linestyle=":", label=f"Ort. = {kpis['genel']['ort. toplam kuyruk']:.2f}")
axs[1].set_xlabel("Zaman")
axs[1].set_ylabel("Toplam Bekleyen")
axs[1].set_title("Toplam Kuyruk Uzunluğu")
axs[1].legend()

# 3. Bekleme süresi histogramı
for c in (GOOD, BAD):
    sel = (players["tier"] == c) & (players["status"] == MATCHED) & (players["arrival"] >= WARMUP)
    w = players["end"][sel] - players["arrival"][sel]
    axs[2].hist(w[w > 0], bins=50, alpha=0.5, label=TIER_NAME[c], density=True)
axs[2].axvline(TAU, color="black", linestyle="--", label=f"Eşik $\\tau={TAU}$")
axs[2].set_xlabel("Bekleme Süresi (w > 0)")
axs[2].set_ylabel("Yoğunluk")
axs[2].set_title("Bekleme Süresi Dağılımı")
axs[2].legend()
plt.tight_layout()
plt.show()

## $\tau$ Duyarlılık Analizi (Trade-off: Bekleme Süresi vs Maç Kalitesi)
$\tau$ değerini 0.5'ten 10'a kadar değiştirerek bekleme süresi ile çapraz maç oranı arasındaki dengeyi inceliyoruz.

In [ ]:
TAU_GRID = [0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 12.0]
results = []

for tau_val in TAU_GRID:
    pl, aq, _ = simulate_phase2_1(lam=LAM, p=P, tau=tau_val, T=30_000, warmup=1_000, seed=SEED)
    res_kpis = compute_kpis_phase2_1(pl, aq, warmup=1_000)
    results.append({
        "tau": tau_val,
        "w_good": res_kpis[TIER_NAME[GOOD]]["ort. bekleme"],
        "w_bad": res_kpis[TIER_NAME[BAD]]["ort. bekleme"],
        "w_all": res_kpis["genel"]["ort. bekleme (genel)"],
        "q_total": res_kpis["genel"]["ort. toplam kuyruk"],
        "cross_ratio": res_kpis["genel"]["capraz eslesme orani (%)"],
    })

print(f"{'tau':>6} | {'Ort. Bekleme (Genel)':>20} | {'Ort. Toplam Kuyruk':>18} | {'Çapraz Eşleşme (%)':>20}")
print("-" * 72)
for r in results:
    print(f"{r['tau']:>6.1f} | {r['w_all']:>20.3f} | {r['q_total']:>18.3f} | {r['cross_ratio']:>19.2f}%")

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
taus = [r["tau"] for r in results]
axs[0].plot(taus, [r["w_all"] for r in results], marker="o", label="Ort. Bekleme")
axs[0].set_xlabel("Çapraz Eşleşme Eşiği ($\\tau$)")
axs[0].set_ylabel("Ortalama Bekleme Süresi")
axs[0].set_title("$\\tau$ vs Bekleme Süresi")
axs[0].grid(True, alpha=0.3)

# Pareto/Trade-off eğrisi: Bekleme Süresi vs Çapraz Eşleşme Oranı
axs[1].plot([r["cross_ratio"] for r in results], [r["w_all"] for r in results], marker="s", color="crimson")
for r in results:
    axs[1].annotate(f"$\\tau={r['tau']}$", (r["cross_ratio"], r["w_all"]), textcoords="offset points", xytext=(5, 5))
axs[1].set_xlabel("Çapraz Eşleşme Oranı (%)")
axs[1].set_ylabel("Ortalama Bekleme Süresi")
axs[1].set_title("Trade-off: Bekleme Süresi vs Çapraz Maç Oranı")
axs[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()